# Fine-Tuning DistilBART on BBC News Summary

This notebook fine-tunes `sshleifer/distilbart-cnn-12-6` (an encoder-decoder summarization model) on the BBC News Summary dataset, then reloads the saved model and runs inference on a new article.

**Flow:**
1. Setup & Installs
2. Configuration
3. Build Dataset from Raw Files
4. Train / Validation / Test Split
5. Tokenization
6. Load Model
7. Train
8. Save Model
9. Reload & Run Inference

## 1. Setup & Installs

In [1]:
!pip install -q transformers datasets evaluate accelerate tensorboard sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch Version:", torch.__version__)
print("Device         :", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch Version: 2.10.0+cu128
Device         : cuda
GPU: Tesla T4


## 2. Configuration

All tunable settings live here so the rest of the notebook never hardcodes them again.

In [4]:
MODEL_NAME = "sshleifer/distilbart-cnn-12-6"

BASE_PATH = "/kaggle/input/datasets/pariza/bbc-news-summary/BBC News Summary"
ARTICLE_FOLDER = os.path.join(BASE_PATH, "News Articles")
SUMMARY_FOLDER = os.path.join(BASE_PATH, "Summaries")
CATEGORIES = ["business", "entertainment", "politics", "sport", "tech"]

NUM_SAMPLES = 500          # quick fine-tuning subset
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128
SEED = 42

OUTPUT_DIR = "./distilbart_bbc"
SAVE_PATH = "./distilbart_bbc_finetuned"


## 3. Build Dataset from Raw Files

Reads each article/summary pair from disk into a single DataFrame.

In [5]:
texts, summaries, categories = [], [], []

for category in CATEGORIES:
    article_path = os.path.join(ARTICLE_FOLDER, category)
    summary_path = os.path.join(SUMMARY_FOLDER, category)

    for file in sorted(os.listdir(article_path)):
        with open(os.path.join(article_path, file), "r", encoding="latin-1") as f:
            article = f.read()

        with open(os.path.join(summary_path, file), "r", encoding="latin-1") as f:
            summary = f.read()

        texts.append(article)
        summaries.append(summary)
        categories.append(category)

df = pd.DataFrame({"Text": texts, "Summary": summaries, "Category": categories})

print("Dataset created successfully! Shape:", df.shape)


Dataset created successfully! Shape: (2225, 3)


In [6]:
# Take only a subset for quick fine-tuning
sample_df = df.sample(n=NUM_SAMPLES, random_state=SEED).reset_index(drop=True)
sample_df = sample_df[["Text", "Summary", "Category"]]

print("Sample dataset shape:", sample_df.shape)
print(sample_df["Category"].value_counts())

sample_df.head()


Sample dataset shape: (500, 3)
Category
business         129
sport            115
tech              92
politics          84
entertainment     80
Name: count, dtype: int64


,Text,Summary,Category
0,UK house prices dip in November\n\nUK house pr...,All areas saw a rise in annual house price inf...,business
1,LSE 'sets date for takeover deal'\n\nThe Londo...,A Â£1.3bn offer from Deutsche Boerse has alrea...,business
2,Harinordoquy suffers France axe\n\nNumber eigh...,Harinordoquy was a second-half replacement in ...,sport
3,Barclays shares up on merger talk\n\nShares in...,Shares in UK banking group Barclays have risen...,business
4,Campaign 'cold calls' questioned\n\nLabour and...,Assistant information commissioner Phil Jones ...,politics


## 4. Train / Validation / Test Split

In [7]:
train_df, temp_df = train_test_split(
    sample_df, test_size=0.20, random_state=SEED, shuffle=True
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training Samples  :", len(train_df))
print("Validation Samples:", len(val_df))
print("Test Samples      :", len(test_df))


Training Samples  : 400
Validation Samples: 50
Test Samples      : 50


In [8]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)


Dataset({
    features: ['Text', 'Summary', 'Category'],
    num_rows: 400
})
Dataset({
    features: ['Text', 'Summary', 'Category'],
    num_rows: 50
})
Dataset({
    features: ['Text', 'Summary', 'Category'],
    num_rows: 50
})


## 5. Tokenization

In [9]:
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)

print("ARTICLE\n")
print(train_dataset[0]["Text"][:500])
print("\nSUMMARY\n")
print(train_dataset[0]["Summary"])


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

ARTICLE

Economy 'stronger than forecast'

The UK economy probably grew at a faster rate in the third quarter than the 0.4% reported, according to Bank of England deputy governor Rachel Lomax.

Private sector business surveys suggest a stronger economy than official estimates, Ms Lomax said. Other surveys collectively show a rapid slowdown in UK house price growth, she pointed out. This means that despite a strong economic growth, base rates will probably stay on hold at 4.75%. Official data comes from t

SUMMARY

"The MPC judges that overall growth was a little higher in the third quarter than the official data currently indicate," Ms Lomax said."Recent work by the Bank has shown that private sector surveys add value, even when preliminary ONS estimates are available," Ms Lomax said in a speech to the North Wales Business Club.Private sector business surveys suggest a stronger economy than official estimates, Ms Lomax said."This year rising oil prices and a significant slowdown in the

Tokenize both the source article and the target summary (via `text_target`), then convert each split to PyTorch tensors.

In [10]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["Text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        text_target=examples["Summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)

train_dataset = train_dataset.remove_columns(["Text", "Summary", "Category"])
val_dataset = val_dataset.remove_columns(["Text", "Summary", "Category"])
test_dataset = test_dataset.remove_columns(["Text", "Summary", "Category"])

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("Training  :", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test      :", len(test_dataset))


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Training  : 400
Validation: 50
Test      : 50


In [11]:
decoded_article = tokenizer.decode(train_dataset[0]["input_ids"], skip_special_tokens=True)
decoded_summary = tokenizer.decode(train_dataset[0]["labels"], skip_special_tokens=True)

print("=" * 60)
print("Decoded Article")
print("=" * 60)
print(decoded_article[:500])

print("\n" + "=" * 60)
print("Decoded Summary")
print("=" * 60)
print(decoded_summary)


Decoded Article
Economy 'stronger than forecast'

The UK economy probably grew at a faster rate in the third quarter than the 0.4% reported, according to Bank of England deputy governor Rachel Lomax.

Private sector business surveys suggest a stronger economy than official estimates, Ms Lomax said. Other surveys collectively show a rapid slowdown in UK house price growth, she pointed out. This means that despite a strong economic growth, base rates will probably stay on hold at 4.75%. Official data comes from t

Decoded Summary
"The MPC judges that overall growth was a little higher in the third quarter than the official data currently indicate," Ms Lomax said."Recent work by the Bank has shown that private sector surveys add value, even when preliminary ONS estimates are available," Ms Lomax said in a speech to the North Wales Business Club.Private sector business surveys suggest a stronger economy than official estimates, Ms Lomax said."This year rising oil prices and a significant s

## 6. Load Model

In [12]:
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(device)

print("Model running on:", device)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model running on: cuda


In [13]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

print("Data collator created")


Data collator created


## 7. Train

In [14]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    learning_rate=2e-5,
    remove_unused_columns=False,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=2,
    warmup_steps=50,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_torch",

    predict_with_generate=True,

    logging_dir="./logs",
    report_to="tensorboard",

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=torch.cuda.is_available(),
    seed=SEED,
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=1)

print("Learning Rate:", training_args.learning_rate)
print("Batch Size   :", training_args.per_device_train_batch_size)
print("Epochs       :", training_args.num_train_epochs)
print("Warmup Steps :", training_args.warmup_steps)
print("Optimizer    :", training_args.optim)
print("Scheduler    :", training_args.lr_scheduler_type)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Learning Rate: 2e-05
Batch Size   : 4
Epochs       : 2
Warmup Steps : 50
Optimizer    : OptimizerNames.ADAMW_TORCH
Scheduler    : SchedulerType.LINEAR


In [15]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[early_stopping],
)


In [16]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,2.244788,1.453861
2,0.988445,0.807658


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=100, training_loss=1.616616439819336, metrics={'train_runtime': 147.3723, 'train_samples_per_second': 5.428, 'train_steps_per_second': 0.679, 'total_flos': 619164834201600.0, 'train_loss': 1.616616439819336, 'epoch': 2.0})

## 8. Save Model

In [17]:
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


## 9. Reload & Run Inference

In [18]:
loaded_tokenizer = BartTokenizer.from_pretrained(SAVE_PATH)
loaded_model = BartForConditionalGeneration.from_pretrained(SAVE_PATH)
loaded_model.to(device)

print("Model reloaded successfully!")


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model reloaded successfully!


In [19]:
new_article = """
Artificial Intelligence is increasingly being used in healthcare to assist doctors
in diagnosing diseases, analyzing medical images, predicting patient outcomes,
and improving treatment planning. Hospitals around the world are adopting AI-based
systems to increase efficiency and provide better patient care.
"""

inputs = loaded_tokenizer(
    new_article,
    return_tensors="pt",
    max_length=MAX_INPUT_LENGTH,
    truncation=True,
).to(device)

summary_ids = loaded_model.generate(
    inputs["input_ids"],
    max_length=100,
    min_length=10,
    num_beams=4,
)

summary = loaded_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Generated Summary:\n")
print(summary)


Generated Summary:

Artificial Intelligence is increasingly being used in healthcare to assist doctors in diagnosing diseases, analyzing medical images, predicting patient outcomes, and improving treatment planning.Artificial Artificial Intelligence is also being used to help doctors and nurses diagnose diseases, analyze medical images and predict patient outcomes. Hospitals around the world are adopting AI-based systems to increase efficiency and provide better patient care.
